# 8.5. Batch Normalization

In the [last](https://github.com/DonaldKellett/my-ascend-notebooks/blob/90a2255c66346c38f16d7fc2e8d763a01592b38e/atomgit-ai/00-d2l-mindspore-ch8-modern-convolutional-neural-networks/03-multi-branch-networks-googlenet.ipynb) and previous chapters, we used [`mindspore.nn.BatchNorm2d`](https://www.mindspore.cn/docs/en/r2.9.0/api_python/nn/mindspore.nn.BatchNorm2d.html#mindspore.nn.BatchNorm2d) layers to stabilize the training of modern CNNs such as GoogLeNet without a solid understanding of what batch normalization is and how it actually works.

In this chapter, we'll take a closer look at the mathematics between batch normalization and understand why its discovery enabled modern deep learning practitioners to reliably train deep networks in excess of 100 layers without encountering vanishing or exploding gradients.

Unlike previous chapters involving modern deep networks which required the use of datacenter-level Ascend 910B4 training-optimized NPUs via the [AtomGit AI Notebook Lab](https://ai.gitcode.com/docs/notebooks/free-usage/) cloud environment to achieve reasonable training times, we'll simply implement our own batch normalization layer from first principles in this chapter and apply it to LeNet from chapter 7, see how it improves the validation loss and accuracy of our model. The entire training process should complete on the [OrangePi AIpro \(20T\)](http://www.orangepi.org/html/hardWare/computerAndMicrocontrollers/details/Orange-Pi-AIpro%2820t%29.html) featuring a single Ascend 310B1 NPU chip and core in around 30 minutes.

The software versions used in this notebook are listed below.

1. Ubuntu 22.04 LTS
1. Python 3.12
1. MindSpore 2.9.0
1. CANN 9.0.0

In [1]:
!npu-smi info

+--------------------------------------------------------------------------------------------------------+
| npu-smi 23.0.0                                   Version: 23.0.0                                       |
+-------------------------------+-----------------+------------------------------------------------------+
| NPU     Name                  | Health          | Power(W)     Temp(C)           Hugepages-Usage(page) |
| Chip    Device                | Bus-Id          | AICore(%)    Memory-Usage(MB)                        |
+===============================+=================+======================================================+
| 0       310B1                 | Alarm           | 0.0          48                15    / 15            |
| 0       0                     | NA              | 0            4831 / 23673                            |
+===============================+=================+======================================================+


In [2]:
!cat requirements.txt

absl-py==2.4.0
attrs==26.1.0
cloudpickle==3.1.2
decorator==5.2.1
jupyterlab==4.5.7
jupyterlab-git==0.53.0
jupyter-resource-usage==1.2.1
loguru==0.7.3
matplotlib==3.10.9
mindspore==2.9.0
ml-dtypes==0.5.4
msguard==0.0.8
openpyxl==3.1.5
opentelemetry-exporter-otlp-proto-grpc==1.33.1
opentelemetry-exporter-otlp-proto-http==1.33.1
pandas~=2.2
plotly>=5.11.0
pydantic==2.13.4
sympy==1.14.0
tornado==6.5.5


In [3]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-9.0.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

MindSpore version:  2.9.0
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 8.5.3. Implementation From Scratch

TODO